In [21]:
#reading pdf files
# !pip install pypdf

# !pip install pydantic

# !pip install fitz
# !pip install pymupdf


AI_RAG_Compliance_Questionnaire

In [22]:
#extration oof text from pdf
import fitz
doc = fitz.open('AI_RAG_Compliance_Questionnaire.pdf')
raw_text_file1 = ""
for page in doc:
   raw_text_file1+=page.get_text()
print(raw_text_file1)

AI / RAG System Compliance Questionnaire
1. Data Privacy & Protection (DPDP / GDPR)
- Do you collect personal data from users?
- Is explicit user consent obtained before data collection?
- Can users request deletion of their data?
- Is sensitive data encrypted at rest and in transit?
- Do you store data within permitted jurisdictions (data localization)?
- Is data anonymized or masked where possible?
- Do you have a privacy policy available to users?
- Is access to user data restricted based on roles?
- Do you log access to sensitive data?
- Do you regularly audit data storage practices?
2. Security & Infrastructure (ISO 27001 / OWASP)
- Are APIs secured with authentication and authorization?
- Is input validation implemented to prevent injection attacks?
- Are secrets (API keys, tokens) securely stored?
- Do you perform regular vulnerability scans?
- Is HTTPS enforced across all endpoints?
- Do you have firewall and intrusion detection systems?
- Are backups taken and tested regularly

In [23]:
# coveting numbers to SECTIONS 

import re

raw_text_file1 = re.sub(r'^\d+\.\s*', 'SECTION: ', raw_text_file1, flags=re.MULTILINE)

In [24]:
print(raw_text_file1)

AI / RAG System Compliance Questionnaire
SECTION: Data Privacy & Protection (DPDP / GDPR)
- Do you collect personal data from users?
- Is explicit user consent obtained before data collection?
- Can users request deletion of their data?
- Is sensitive data encrypted at rest and in transit?
- Do you store data within permitted jurisdictions (data localization)?
- Is data anonymized or masked where possible?
- Do you have a privacy policy available to users?
- Is access to user data restricted based on roles?
- Do you log access to sensitive data?
- Do you regularly audit data storage practices?
SECTION: Security & Infrastructure (ISO 27001 / OWASP)
- Are APIs secured with authentication and authorization?
- Is input validation implemented to prevent injection attacks?
- Are secrets (API keys, tokens) securely stored?
- Do you perform regular vulnerability scans?
- Is HTTPS enforced across all endpoints?
- Do you have firewall and intrusion detection systems?
- Are backups taken and test

In [ ]:
structured_data = []
current_section = None

lines = raw_text_file1.split("\n")

for line in lines:
    line = line.strip()

    if not line:
        continue

    # Detect SECTION 
    if "SECTION:" in line:
        section_name = line.split("SECTION:")[-1].strip()

        current_section = {
            "section": section_name,
            "content": []
        }

        structured_data.append(current_section)

    # Detect questions
    elif line.startswith("-"):
        if current_section is not None:
            question = line.lstrip("- ").strip()
            current_section["content"].append(question)

# Print cleanly
from pprint import pprint
pprint(structured_data)

[{'content': ['Do you collect personal data from users?',
              'Is explicit user consent obtained before data collection?',
              'Can users request deletion of their data?',
              'Is sensitive data encrypted at rest and in transit?',
              'Do you store data within permitted jurisdictions (data '
              'localization)?',
              'Is data anonymized or masked where possible?',
              'Do you have a privacy policy available to users?',
              'Is access to user data restricted based on roles?',
              'Do you log access to sensitive data?',
              'Do you regularly audit data storage practices?'],
  'section': 'Data Privacy & Protection (DPDP / GDPR)'},
 {'content': ['Are APIs secured with authentication and authorization?',
              'Is input validation implemented to prevent injection attacks?',
              'Are secrets (API keys, tokens) securely stored?',
              'Do you perform regular vulnerabi

In [26]:
chunks = []

# Example: your structured_data from previous step
# structured_data = [
#   {"section": "...", "content": ["Q1", "Q2", ...]}
# ]

for item in structured_data:
    section = item["section"]
    questions = item["content"]

    # 🔹 Step 1: Convert list of questions → single text
    combined_text = section + ":\n" + "\n".join(questions)

    # 🔹 Step 2: Assign type based on section name
    if "Privacy" in section:
        doc_type = "privacy"
    elif "Security" in section:
        doc_type = "security"
    elif "AI" in section or "LLM" in section:
        doc_type = "ai-risk"
    elif "Logging" in section:
        doc_type = "logging"
    elif "Banking" in section or "RBI" in section:
        doc_type = "banking"
    else:
        doc_type = "general"

    # 🔹 Step 3: Create chunk
    chunk = {
        "text": combined_text,
        "section": section,
        "source": "AI_RAG_Compliance_Questionnaire.pdf",
        "type": doc_type
    }

    chunks.append(chunk)

# Preview
from pprint import pprint
pprint(chunks)

[{'section': 'Data Privacy & Protection (DPDP / GDPR)',
  'source': 'AI_RAG_Compliance_Questionnaire.pdf',
  'text': 'Data Privacy & Protection (DPDP / GDPR):\n'
          'Do you collect personal data from users?\n'
          'Is explicit user consent obtained before data collection?\n'
          'Can users request deletion of their data?\n'
          'Is sensitive data encrypted at rest and in transit?\n'
          'Do you store data within permitted jurisdictions (data '
          'localization)?\n'
          'Is data anonymized or masked where possible?\n'
          'Do you have a privacy policy available to users?\n'
          'Is access to user data restricted based on roles?\n'
          'Do you log access to sensitive data?\n'
          'Do you regularly audit data storage practices?',
  'type': 'privacy'},
 {'section': 'Security & Infrastructure (ISO 27001 / OWASP)',
  'source': 'AI_RAG_Compliance_Questionnaire.pdf',
  'text': 'Security & Infrastructure (ISO 27001 / OWASP):\n'

In [ ]:
import json

with open("chunks1.json", "w") as f:
    json.dump(chunks, f, indent=2)

Rib files have same structure we will just write function use it for all
1. Master_Direction-Information_Technology_Framework_for_the_NBFC_Sector.pdf
2. NT41802062016.pdf
3. RBI_Master_Direction_dated_18.02.2021_-_Digital_Payment_Security_Controls.pdf

In [1]:
import fitz
import re

def process_rbi_pdf(file_path):
    # 🔹 Step 1: Extract text
    doc = fitz.open(file_path)
    text = ""

    for page in doc:
        page_text = page.get_text()
        if page_text:
            text += page_text + "\n\n"

    # 🔹 Step 2: Clean text
    text = re.sub(r'Automatic Zoom', '', text)
    text = re.sub(r'\n\s*\d+\s*\n', '\n', text)  # remove page numbers
    text = re.sub(r'\n+', '\n', text)  # remove extra newlines

    # 🔹 Step 3: Convert numbered sections → SECTION
    text = re.sub(r'^\d+\.\s*', 'SECTION: ', text, flags=re.MULTILINE)

    # 🔹 Step 4: Structure (section → content)
    structured_data = []
    current_section = None

    lines = text.split("\n")

    for line in lines:
        line = line.strip()

        if not line:
            continue

        # Detect section
        if "SECTION:" in line:
            section_name = line.split("SECTION:")[-1].strip()

            current_section = {
                "section": section_name,
                "content": []
            }
            structured_data.append(current_section)

        # Detect bullet points OR normal lines
        elif current_section:
            if line.startswith("-"):
                question = line.lstrip("- ").strip()
            else:
                question = line.strip()

            current_section["content"].append(question)

    # 🔹 Step 5: Chunking (section = 1 chunk)
    chunks = []

    for item in structured_data:
        section = item["section"]
        content = item["content"]

        if not content:
            continue

        combined_text = section + ":\n" + "\n".join(content)

        # Assign type
        if "security" in section.lower():
            doc_type = "security"
        elif "risk" in section.lower():
            doc_type = "risk"
        elif "data" in section.lower():
            doc_type = "data"
        else:
            doc_type = "banking"

        chunk = {
            "text": combined_text,
            "section": section,
            "source": "RBI",
            "file_name": file_path.split("/")[-1],
            "type": doc_type
        }

        chunks.append(chunk)

    return chunks

In [3]:
import json

rbi_files = [
    "../ComplianceIQ/data/Master_Direction-Information_Technology_Framework_for_the_NBFC_Sector.pdf",
    "../ComplianceIQ/data/NT41802062016.pdf",
    "../ComplianceIQ/data/RBI_Master_Direction_dated_18.02.2021_-_Digital_Payment_Security_Controls.pdf"
]

for i, file in enumerate(rbi_files, start=1):
    chunks = process_rbi_pdf(file)

    file_name = f"chunk{i}.json"

    with open(file_name, "w") as f:
        json.dump(chunks, f, indent=2)

    print(f"Saved {file_name} with {len(chunks)} chunks")

Saved chunk1.json with 69 chunks
Saved chunk2.json with 141 chunks
Saved chunk3.json with 72 chunks


GDPR text file

In [4]:
import re

def process_gdpr_txt(file_path):
    # 🔹 Step 1: Read file
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    # 🔹 Step 2: Clean text
    text = re.sub(r'\n+', '\n', text)  # remove extra newlines
    text = text.strip()

    # 🔹 Step 3: Split by clauses ( (1), (2), (3)... )
    clauses = re.split(r'\(\d+\)', text)

    # Remove first part (before (1)) if empty or header
    clauses = clauses[1:]

    chunks = []

    for i, clause in enumerate(clauses, start=1):
        clause_text = clause.strip()

        if not clause_text:
            continue

        # 🔹 Step 4: Create chunk
        chunk = {
            "text": f"Clause ({i}): {clause_text}",
            "section": f"Clause {i}",
            "source": "GDPR",
            "file_name": file_path.split("/")[-1],
            "type": "privacy"
        }

        chunks.append(chunk)

    return chunks

In [6]:
import json

gdpr_chunks = process_gdpr_txt("../ComplianceIQ/data/GDPR.txt")

with open("chunk4.json", "w") as f:
    json.dump(gdpr_chunks, f, indent=2)

print(f"Saved chunk4.json with {len(gdpr_chunks)} chunks")

Saved chunk4.json with 408 chunks


In [8]:
import pandas as pd
import json

# 🔹 Function 1: Employee CSV
def process_employee_csv(file_path):
    df = pd.read_csv(file_path)
    chunks = []

    for _, row in df.iterrows():
        text = (
            f"Employee {row['Name']} worked {row['Working_Days']} days in {row['Month']}. "
            f"Target sales were {row['Target_Sales']} and actual sales were {row['Actual_Sales']}. "
            f"Customer satisfaction score was {row['Customer_Satisfaction_Score']}. "
            f"Policy compliance was {row['Policy_Compliance']}. "
            f"Issues: {row['Non_Compliance_Reason']}."
        )

        chunk = {
            "text": text,
            "source": "employee_data.csv",
            "type": "employee",
            "employee_id": int(row["Employee_ID"])
        }

        chunks.append(chunk)

    return chunks

In [17]:
# 🔹 Function 2: Economic CSV
def process_economic_csv(file_path):
    df = pd.read_csv(file_path)
    chunks = []

    for _, row in df.iterrows():
        text = (
            f"In {row['Periods']}, employment jobs were {row['Employment Jobs']} thousand. "
            f"Labour volume was {row['Labour volume']} thousand. "
            f"Hourly wage was {row['Wage per job Hourly wage (euro)']} euros. "
            f"Monthly wage including overtime was {row['Wage per job Monthly wages Monthly wage including overtime (euro)']} euros. "
            f"Yearly wage including bonuses was {row['Yearly wage Yearly wage including bonuses (euro)']} euros."
        )

        chunk = {
            "text": text,
            "source": "economic_data.csv",
            "type": "economic",
            "period": row["Periods"]
        }

        chunks.append(chunk)

    return chunks

In [11]:

employee_chunks = process_employee_csv("../ComplianceIQ/data/Policy_Compliance_Dataset_Updated.csv")

with open("chunk5.json", "w") as f:
    json.dump(employee_chunks, f, indent=2)

print(f"Saved chunk5.json with {len(employee_chunks)} chunks")




Saved chunk5.json with 4000 chunks


In [19]:
# Economic CSV → chunk6.json
economic_chunks = process_economic_csv("../ComplianceIQ/data/table__81434ENG.csv")

with open("chunk6.json", "w") as f:
    json.dump(economic_chunks, f, indent=2)

print(f"Saved chunk6.json with {len(economic_chunks)} chunks")

Saved chunk6.json with 171 chunks
